In [1]:
#EDA mapped files

In [3]:
import pandas as pd

file1 = "mapped-grote.xlsx"
file2 = "mapped-dayton.xlsx"

# Load only the Digital Assets sheet
df1 = pd.read_excel(file1, sheet_name="Digital_Assets")
df2 = pd.read_excel(file2, sheet_name="Digital_Assets")


In [5]:
df1.head()

,Part Number,Digital Change Type,MediaType,FileName,FilePath,FileType,Representation,Orientation,Height,Width
0,00200,A,P04,00200.jpg,NaN,JPG,A,FRO,1500.0,1500.0
1,00210,A,P04,00210.jpg,NaN,JPG,A,FRO,1500.0,1500.0
2,00210,A,DAS,Brite_Zone_Data_Sheet_Rectangular_work_lamps.pdf,NaN,PDF,NaN,NaN,NaN,NaN
3,00210,A,DAS,Brite_Zone_Data_Sheet_Rectangular_work_lamp_FR...,NaN,PDF,NaN,NaN,NaN,NaN
4,00210,A,DAS,Brite_Zone_Data_Sheet_Rectangular_work_lamp_SP...,NaN,PDF,NaN,NaN,NaN,NaN


In [6]:
missing_1 = df1.isna().sum().sort_values(ascending=False)
missing_2 = df2.isna().sum().sort_values(ascending=False)

missing_1, missing_2


(FilePath               7823
 Orientation            4675
 Representation         3262
 Height                 3046
 Width                  3046
 Part Number               0
 Digital Change Type       0
 FileType                  0
 MediaType                 0
 FileName                  0
 dtype: int64,
 FilePath               8727
 Representation         3364
 Orientation            3358
 FileType                635
 Height                  277
 Width                   277
 Digital Change Type       0
 Part Number               0
 MediaType                 0
 FileName                  0
 dtype: int64)

In [7]:
missing_pct_1 = (df1.isna().mean() * 100).round(2)
missing_pct_2 = (df2.isna().mean() * 100).round(2)

missing_pct_1, missing_pct_2


(Part Number              0.00
 Digital Change Type      0.00
 MediaType                0.00
 FileName                 0.00
 FilePath               100.00
 FileType                 0.00
 Representation          41.70
 Orientation             59.76
 Height                  38.94
 Width                   38.94
 dtype: float64,
 Part Number              0.00
 Digital Change Type      0.00
 MediaType                0.00
 FileName                 0.00
 FilePath               100.00
 FileType                 7.28
 Representation          38.55
 Orientation             38.48
 Height                   3.17
 Width                    3.17
 dtype: float64)

In [8]:
critical_cols = [
    "Part Number",
    "MediaType",
    "FileName",
    "FilePath",
    "FileType",
    "Height",
    "Width",
    "Orientation"
]

df1[critical_cols].isna().sum()


Part Number       0
MediaType         0
FileName          0
FilePath       7823
FileType          0
Height         3046
Width          3046
Orientation    4675
dtype: int64

In [9]:
df1[df1["MediaType"].isna()]


,Part Number,Digital Change Type,MediaType,FileName,FilePath,FileType,Representation,Orientation,Height,Width


In [10]:
comparison = pd.DataFrame({
    "missing_file_1": missing_pct_1,
    "missing_file_2": missing_pct_2
}).fillna(0)

comparison.sort_values(by="missing_file_1", ascending=False)


,missing_file_1,missing_file_2
FilePath,100.00,100.00
Orientation,59.76,38.48
Representation,41.70,38.55
Height,38.94,3.17
Width,38.94,3.17
Part Number,0.00,0.00
Digital Change Type,0.00,0.00
FileType,0.00,7.28
MediaType,0.00,0.00
FileName,0.00,0.00


In [11]:
blocking_cols = ["MediaType", "FileName", "FilePath"]

df1["is_blocking"] = df1[blocking_cols].isna().any(axis=1)
df1["is_blocking"].value_counts()


is_blocking
True    7823
Name: count, dtype: int64

Normalize FileType first

In [12]:
df = df1.copy()  # or df2

df["FileType_norm"] = (
    df["FileType"]
    .astype(str)
    .str.lower()
    .str.replace(".", "", regex=False)
)


Define file type groups

In [13]:
IMAGE_TYPES = ["jpg", "jpeg", "png", "gif"]
PDF_TYPES   = ["pdf"]


️ Filter PDFs

In [14]:
df_pdf = df[df["FileType_norm"].isin(PDF_TYPES)]

df_pdf.shape


(3046, 12)

Filter images (jpg, png, gif)

In [16]:
df_img = df[df["FileType_norm"].isin(IMAGE_TYPES)]

df_img.shape


(4777, 12)

Missing values in images

In [17]:
df_img.isna().sum()


Part Number               0
Digital Change Type       0
MediaType                 0
FileName                  0
FilePath               4777
FileType                  0
Representation          216
Orientation            1629
Height                    0
Width                     0
is_blocking               0
FileType_norm             0
dtype: int64

Critical checks per asset type
- PDFs → usually should NOT have dimensions

In [18]:
pdf_dimension_issues = df_pdf[
    df_pdf[["Height", "Width"]].notna().any(axis=1)
]

pdf_dimension_issues


,Part Number,Digital Change Type,MediaType,FileName,FilePath,FileType,Representation,Orientation,Height,Width,is_blocking,FileType_norm


Images → MUST have dimensions

In [19]:
img_dimension_issues = df_img[
    df_img[["Height", "Width"]].isna().any(axis=1)
]

img_dimension_issues


,Part Number,Digital Change Type,MediaType,FileName,FilePath,FileType,Representation,Orientation,Height,Width,is_blocking,FileType_norm


MediaType vs FileType consistency

In [21]:
media_mismatch = df[
    (
        (df["FileType_norm"].isin(IMAGE_TYPES)) &
        (df["MediaType"].str.lower() != "image")
    ) |
    (
        (df["FileType_norm"] == "pdf") &
        (df["MediaType"].str.lower() != "document")
    )
]

media_mismatch


,Part Number,Digital Change Type,MediaType,FileName,FilePath,FileType,Representation,Orientation,Height,Width,is_blocking,FileType_norm
0,00200,A,P04,00200.jpg,NaN,JPG,A,FRO,1500.0,1500.0,True,jpg
1,00210,A,P04,00210.jpg,NaN,JPG,A,FRO,1500.0,1500.0,True,jpg
2,00210,A,DAS,Brite_Zone_Data_Sheet_Rectangular_work_lamps.pdf,NaN,PDF,NaN,NaN,NaN,NaN,True,pdf
3,00210,A,DAS,Brite_Zone_Data_Sheet_Rectangular_work_lamp_FR...,NaN,PDF,NaN,NaN,NaN,NaN,True,pdf
4,00210,A,DAS,Brite_Zone_Data_Sheet_Rectangular_work_lamp_SP...,NaN,PDF,NaN,NaN,NaN,NaN,True,pdf
...,...,...,...,...,...,...,...,...,...,...,...,...
7818,PGT6710NPG,A,P04,PGT6710NPG.jpg,NaN,JPG,A,FRO,1500.0,1500.0,True,jpg
7819,STT5000RPG,A,P04,STT5000RPG.jpg,NaN,JPG,A,FRO,1500.0,1500.0,True,jpg
7820,STT5100RPG,A,P04,STT5100RPG.jpg,NaN,JPG,A,FRO,1500.0,1500.0,True,jpg
7821,STT5110RPG,A,P04,STT5110RPG.jpg,NaN,JPG,NaN,NaN,825.0,825.0,True,jpg


Summary counts

In [22]:
summary = pd.DataFrame({
    "count": df["FileType_norm"].value_counts(),
    "missing_mediatype": df.groupby("FileType_norm")["MediaType"].apply(lambda x: x.isna().sum()),
    "missing_height": df.groupby("FileType_norm")["Height"].apply(lambda x: x.isna().sum()),
    "missing_width": df.groupby("FileType_norm")["Width"].apply(lambda x: x.isna().sum())
})

summary


,count,missing_mediatype,missing_height,missing_width
FileType_norm,,,,
gif,1413,0,0,0
jpg,3364,0,0,0
pdf,3046,0,3046,3046


In [23]:
df["image_etl_block"] = (
    df["FileType_norm"].isin(IMAGE_TYPES) &
    df[["MediaType", "FileName", "FilePath", "Height", "Width"]].isna().any(axis=1)
)

df["image_etl_block"].value_counts()


image_etl_block
True     4777
False    3046
Name: count, dtype: int64